In [ ]:
from db.conf import create_db_engine, get_async_session
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
engine = create_db_engine()
db = get_async_session(engine)

In [ ]:
from utils.common import var_or_exception
from core.transcribe import FileAudioData, LemonfoxClient
from core.video import FileVideoData, ClipTaggerClient
from core.file import FilesystemVideoSource, VideoFile, AudioFile
import os
from core.processors.common import PostDetails
from uuid import UUID

from db.repositories.videos import VideoRepository

async with db() as session:
  video_repo = VideoRepository(session)
  video = await video_repo.get_video_by_id(
      UUID("00196292-b888-11f0-a920-0b840d3c65b5"),
      with_annotations=True,
      with_meta=True,
      with_scraped_data=True
  )

last_scraped_data = max(video.scraped_data, key=lambda d: d.created_at)
post_data = PostDetails(**last_scraped_data.data)

LOCAL_VIDEO_STORAGE_PATH = var_or_exception("LOCAL_VIDEO_STORAGE_PATH")
file_url = os.path.join(LOCAL_VIDEO_STORAGE_PATH, post_data.download_url.replace("storage://", ""))
video_source = FilesystemVideoSource(file_url)

video_file = VideoFile(video_source)
audio_file = AudioFile(video_source)

clip_tagger_client = ClipTaggerClient()
lemonfox_client = LemonfoxClient()

video_data = FileVideoData(clip_tagger_client, video_file)
audio_data = FileAudioData(lemonfox_client, audio_file)

In [ ]:
from core.agents.challenge import ChallengeVideoAnalyzer
from core.agents.common import gemini_2_5_flash_lite, gemini_2_5_settings, TemplateManager

agent = ChallengeVideoAnalyzer(gemini_2_5_flash_lite(), gemini_2_5_settings(100), TemplateManager())

In [ ]:
from core.agents.challenge import ChallengeVideoAnalyzerRun

run_input = ChallengeVideoAnalyzerRun(
    challenge="Find a video with a white cat",
    max_frames=4,
    post=post_data,
)

result = await agent.run(run_input, video_data, audio_data)
result